# AnimalDex BioCLIP fine-tuning on Colab

Before running this notebook, use Google Drive for Desktop (or the Drive website) to upload these local items to `MyDrive/AnimalDex/colab_input/`:

- `animaldex_colab_data.zip` — the ZIP of the complete `ml/data` directory, including `raw/` and `splits/`.
- `best.pt` — `ml/runs/bioclip_finetune_fresh/best.pt` (optional; upload it to resume from epoch 1).

In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another GPU) before executing the cells. The data is copied from Drive to Colab's local SSD before training; do not train directly against mounted Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi


In [ ]:
# Clone the committed AnimalDex source and install only the CUDA training dependencies.
!rm -rf /content/AnimalDex
!git clone https://github.com/abhay784/AnimalDex.git /content/AnimalDex
%cd /content/AnimalDex
!pip -q install 'open-clip-torch>=3.3,<4' 'huggingface-hub>=0.36,<1' 'Pillow>=10.4,<12'
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime, then rerun this cell.'
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
# Extract the corpus onto Colab's fast local disk. Do not train directly against mounted Drive.
DRIVE_INPUT = '/content/drive/MyDrive/AnimalDex/colab_input'
!test -f $DRIVE_INPUT/animaldex_colab_data.zip || (echo 'Missing animaldex_colab_data.zip — upload it from your Mac first.' && false)
!rm -rf /content/AnimalDex/ml/data
!unzip -q $DRIVE_INPUT/animaldex_colab_data.zip -d /content/AnimalDex/ml/
!find /content/AnimalDex/ml/data/splits -maxdepth 1 -type f -print


In [ ]:
# Resume from the Mac's epoch-1 result when available.
!mkdir -p /content/AnimalDex/ml/runs/bioclip_colab
!if [ -f $DRIVE_INPUT/best.pt ]; then cp $DRIVE_INPUT/best.pt /content/AnimalDex/ml/runs/bioclip_colab/best.pt; fi
!ls -lh /content/AnimalDex/ml/runs/bioclip_colab || true


In [ ]:
# T4-safe starting configuration. Try --batch-size 32 after a successful run if GPU memory permits.
%cd /content/AnimalDex
!RESUME_ARGS=''; if [ -f ml/runs/bioclip_colab/best.pt ]; then RESUME_ARGS='--resume-checkpoint ml/runs/bioclip_colab/best.pt'; fi; python -u ml/train_bioclip.py --epochs 10 --batch-size 16 --workers 4 --prefetch-factor 2 --unfreeze-blocks 2 --amp --patience 3 --log-interval 50 --device cuda --run-dir ml/runs/bioclip_colab $RESUME_ARGS


In [ ]:
# Persist the current checkpoint and metrics. Run this after every completed epoch or before ending a session.
!mkdir -p $DRIVE_INPUT/results
!rsync -a --info=progress2 /content/AnimalDex/ml/runs/bioclip_colab/ $DRIVE_INPUT/results/
!ls -lh $DRIVE_INPUT/results
